# Part A: Dataset Understanding, Profiling and Data Curation
**Project:** Food Safety and Restaurant Inspection Analytics &nbsp;|&nbsp; **Student:** Rushikesh Maddewad &nbsp;|&nbsp; PRN 202502040037 &nbsp;|&nbsp; Roll No 353

## 1. Dataset source documentation
| Item | Value |
|---|---|
| Dataset | Food Inspections |
| Publisher | City of Chicago / Chicago Department of Public Health |
| Official link | https://data.cityofchicago.org/Health-Human-Services/Food-Inspections/4ijn-s7e5 |
| Access method | Socrata API of the City of Chicago Data Portal (`/resource/4ijn-s7e5.csv`) |
| Extraction date | 29 October 2026 |
| Selected period | 1 January 2022 to 31 December 2025 (the period prescribed in the project document) |
| Filter applied | `inspection_date between '2022-01-01' and '2025-12-31'` |
| Enrichment sources | None (no external dataset was joined) |

The source notes that duplicate reports may remain and that results describe conditions at the time of inspection.

In [1]:
import sys, hashlib
sys.path.insert(0, "../etl")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import curation as cur

pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 60)
RAW = "../data/raw/food_inspections_2022_2025.csv"
raw = cur.load_raw(RAW)
print("Rows:", f"{len(raw):,}", "| Columns:", len(raw.columns))
print("SHA-256 of the downloaded file:", hashlib.sha256(open(RAW, "rb").read()).hexdigest())
print("Date range:", pd.to_datetime(raw["inspection_date"]).min().date(), "to", pd.to_datetime(raw["inspection_date"]).max().date())
print("\nColumns used:", list(raw.columns))
raw.head(3)

Rows: 73,630 | Columns: 17


SHA-256 of the downloaded file: 11dd8f5c6cb6d5baf35de583134068ae6fd02ac5de5c6ae98d63e0d92b510b84
Date range: 2022-01-03 to 2025-12-31

Columns used: ['inspection_id', 'dba_name', 'aka_name', 'license_', 'facility_type', 'risk', 'address', 'city', 'state', 'zip', 'inspection_date', 'inspection_type', 'results', 'violations', 'latitude', 'longitude', 'location']


,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2546078,PULASKI FOOD ENTERPRISES,THE OCKY WAY TOBACCO AND FOOD GRILL,2827416,Grocery Store,Risk 1 (High),2 S PULASKI RD,CHICAGO,IL,60624,2022-01-03T00:00:00.000,License,Not Ready,NaN,41.8806769908738,-87.72581612691941,"\n, \n(41.8806769908738, -87.72581612691941)"
1,2546079,AVLI TAVERNA,AVLI TAVERNA,2616777,Restaurant,Risk 1 (High),1335 W WRIGHTWOOD AVE,CHICAGO,IL,60614,2022-01-04T00:00:00.000,Complaint,Pass w/ Conditions,22. PROPER COLD HOLDING TEMPERATURES - Comments: FOUND TCS FOOD ST...,41.92875605419546,-87.66237147194119,"(41.92875605419546, -87.66237147194119)"
2,2546080,DISCO TACOS,DISCO TACOS,2827150,Restaurant,Risk 1 (High),1605 W MONTROSE AVE,CHICAGO,IL,60613,2022-01-04T00:00:00.000,License,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, ...",41.96147827109243,-87.66949202485075,"(41.96147827109243, -87.66949202485075)"


## 2. Profiling before cleaning

In [2]:
print("Missing values per column:")
cur.missing_profile(raw)

Missing values per column:


,missing,missing_pct,distinct
inspection_id,0,0.00,73630
dba_name,0,0.00,15159
aka_name,75,0.10,14575
license_,3,0.00,19274
facility_type,321,0.44,217
risk,17,0.02,4
address,0,0.00,22095
city,52,0.07,36
state,25,0.03,5
zip,6,0.01,86


In [3]:
print("Result values:");         print(raw["results"].value_counts(dropna=False).to_string())
print("\nRisk values:");          print(raw["risk"].value_counts(dropna=False).to_string())
print("\nState values:");         print(raw["state"].value_counts(dropna=False).head(6).to_string())
print("\nCity spellings:");       print(raw["city"].value_counts(dropna=False).head(6).to_string())
print("\nDistinct raw facility types:", raw["facility_type"].nunique())
print("Distinct inspection types :", raw["inspection_type"].nunique())
dates = pd.to_datetime(raw["inspection_date"])
print("\nTime of day present?", bool((dates.dt.hour + dates.dt.minute + dates.dt.second).any()))
lat, lon = pd.to_numeric(raw["latitude"]), pd.to_numeric(raw["longitude"])
print("Latitude range :", lat.min(), lat.max(), "| Longitude range:", lon.min(), lon.max())

Result values:
results
Pass                    37749
Fail                    14261
Pass w/ Conditions       9744
Out of Business          5304
No Entry                 4991
Not Ready                1562
Business Not Located       19

Risk values:
risk
Risk 1 (High)      59168
Risk 2 (Medium)    10874
Risk 3 (Low)        3533
All                   38
NaN                   17

State values:
state
IL     73592
NaN       25
IN         8
CA         3
CO         1
WI         1

City spellings:
city
CHICAGO     73317
Chicago       117
NaN            52
chicago        48
CCHICAGO       13
EVANSTON       11

Distinct raw facility types: 217
Distinct inspection types : 15

Time of day present? False


Latitude range : 41.64467013219805 42.02106424782547 | Longitude range: -87.90687413400109 -87.5250941359867


## 3. Curation
All rules live in `etl/curation.py` so the same logic is reused by the ETL. Every decision is attribute-specific, and the number of affected rows is calculated by the code (nothing is typed in by hand).

In [4]:
curated, dq = cur.curate(raw)
dq.rename(columns={"problem": "Quality problem", "field": "Field", "count": "Count", "treatment": "Treatment", "justification": "Justification"})

,Quality problem,Field,Count,Treatment,Justification
0,Invalid timestamp,inspection_date,0,Exclude (none found),A date that cannot be parsed cannot be placed in DimDate.
1,Exact duplicate record,all columns,0,Remove,Identical rows add no information.
2,Repeated inspection_id,inspection_id,0,Investigate (none found),inspection_id is the source identifier of one inspection.
3,"Logical duplicate (same licence, date, type, result and violation ...",inspection_id,29,Remove later copies,The source documentation says duplicate reports may remain; identi...
4,"Same licence, date and type but different result/violations",license_number,496,Retain and flag,Different content means they may be separate visits (e.g. morning ...
5,Result value not in documented list,results,0,Map to Not Completed (none found),Results are checked against the source documentation.
6,Invalid risk code 'All',risk,38,Map to Unknown member,"Only Risk 1/2/3 are documented, so 'All' is not a real category."
7,Missing risk,risk,17,Map to Unknown member,Facts are kept; excluding them would bias counts.
8,Missing facility type,facility_type,320,Map to Unknown member,Facts are kept; type is not needed for counts.
9,Free-text facility types (distinct raw spellings),facility_type,217,Title-case + keyword grouping into FacilityGroup,Raw type is free text with spelling variants; the group is a DERIV...


**Inference (data-quality findings):**
- **Structurally clean data:** there are no exact duplicates, no repeated `inspection_id`, no unparsable dates and no coordinates outside Chicago.
- **Duplicates:** 29 repeated reports (same licence, date, type, result and violation text but a different `inspection_id`) were removed. A further 496 rows share licence, date and type but have different content, so they could be separate visits; they are kept and flagged.
- **Violation text is the biggest gap and depends on the result:** 10,823 *Pass* inspections have no text, which means no violation was cited, so their count is 0. 552 *Fail* or *Pass w/ Conditions* inspections have no text, which is inconsistent, so their count is left **unknown (NULL)** instead of a wrong 0.
- **Codes:** 38 rows have the undocumented risk code `All` and 17 have no risk; both go to an explicit *Unknown* member (55 rows in total). 320 rows have no facility type and 6 no ZIP.
- **Not completed results:** 11,864 rows (16%) are *Out of Business*, *No Entry*, *Not Ready* or *Business Not Located*. No assessment took place, so they stay in staging but are not loaded as inspections in the fact table.
- **No time of day:** every timestamp is 00:00:00, so an hourly pattern cannot be analysed and is not invented.
- **Outliers:** 2,686 inspections have unusually many violations (above Q3 + 1.5 IQR). They are kept and flagged because they are exactly the badly failing premises that a food-safety team wants to find.

## 4. Reconciliation of row counts and key measures before and after curation

In [5]:
before = raw["results"].value_counts()
after = curated["results"].value_counts()
rec = pd.DataFrame({"Before curation": before, "After curation": after}).fillna(0).astype(int)
rec["Removed"] = rec["Before curation"] - rec["After curation"]
rec.loc["TOTAL"] = rec.sum()
display(rec)

comp = curated[curated["is_completed"]]
print("Completed inspections (FactInspection grain):", f"{len(comp):,}")
print("Failed inspections:", int(comp["is_failed_inspection"].sum()))
print("Total violations (known counts):", int(comp["violation_count"].sum()))
print("Rows removed overall:", len(raw) - len(curated), "(= logical duplicates)")

,Before curation,After curation,Removed
results,,,
Pass,37749,37734,15
Fail,14261,14260,1
Pass w/ Conditions,9744,9743,1
Out of Business,5304,5294,10
No Entry,4991,4991,0
Not Ready,1562,1560,2
Business Not Located,19,19,0
TOTAL,73630,73601,29


Completed inspections (FactInspection grain): 61,737
Failed inspections: 14260
Total violations (known counts): 230929
Rows removed overall: 29 (= logical duplicates)


## 5. Derived attributes (documentation)
Derived categories are **not** source fields. Each formula, threshold and mapping is listed here and in `docs/data_dictionary.md`.

In [6]:
pd.DataFrame(cur.DERIVED_ATTRIBUTES, columns=["Derived attribute", "Formula", "Threshold / code mapping", "External enrichment"])

,Derived attribute,Formula,Threshold / code mapping,External enrichment
0,Year,YEAR(inspection_date),-,None
1,Quarter,QUARTER(inspection_date),1-4,None
2,Month,MONTH(inspection_date),1-12,None
3,DayOfWeek,Day name of inspection_date,Monday-Sunday,None
4,IsWeekend,"DayOfWeek in (Saturday, Sunday)",TRUE/FALSE,None
5,ViolationCount,Number of numbered items ('34. FLOORS ...') in the Violations text,Completed inspections only. Pass without text = 0. Fail / Pass w/ ...,None
6,HasViolations,ViolationCount > 0,NULL when ViolationCount is NULL,None
7,ResultGroup,Mapping of Results,Pass -> Pass; Pass w/ Conditions -> Conditional Pass; Fail -> Fail...,None
8,RiskGroup,Mapping of Risk,Risk 1 (High) -> High; Risk 2 (Medium) -> Medium; Risk 3 (Low) -> ...,None
9,FacilityGroup,Ordered keyword rules on lower-cased Facility Type,Restaurant & Food Service; Schools & Childcare; Healthcare & Resid...,None (rules are in etl/curation.py)


In [7]:
print("FacilityGroup coverage:")
display(curated["facility_group"].value_counts().to_frame("rows").assign(pct=lambda d: (d.rows / d.rows.sum() * 100).round(2)))
print("Facility types that fall into 'Other':")
print(curated.loc[curated["facility_group"] == "Other", "facility_type_clean"].value_counts().to_string())

FacilityGroup coverage:


,rows,pct
facility_group,,
Restaurant & Food Service,52217,70.95
Schools & Childcare,9193,12.49
Retail Food & Grocery,8626,11.72
"Mobile, Catering & Events",1822,2.48
Healthcare & Residential Care,1404,1.91
Unknown,320,0.43
Other,19,0.03


Facility types that fall into 'Other':
facility_type_clean
Other                                     9
Regulated Business                        4
A-Not-For-Profit Chef Training Program    3
Greenhouse                                1
Unlicensed Facility                       1
Limited Business                          1


**Inference:** the keyword rules place all but a handful of rows into a real group. Only the rows listed under *Other* (a few unusual free-text types) are unclassified, so FacilityGroup is reliable for analysis. Rows without a facility type are shown as *Unknown* and not forced into a group.

## 6. Save the curated data (input of staging and of the EDA)

In [8]:
curated.to_csv("../data/curated/inspections_curated.csv", index=False)
print("Saved", f"{len(curated):,}", "curated rows to data/curated/inspections_curated.csv")

# data-quality report as a markdown file (part of the project deliverables)
lines = ["# Data-Quality Report", "",
         "Source: City of Chicago Food Inspections, 1 Jan 2022 to 31 Dec 2025 (extracted 29 Oct 2026). Every count below is computed by `etl/curation.py` (see `notebooks/01_data_curation.ipynb`).", "",
         f"Raw rows: {len(raw):,}. Curated rows: {len(curated):,}. Completed inspections (FactInspection grain): {int(curated['is_completed'].sum()):,}.", "",
         "| Quality problem | Field | Count | Treatment | Justification |", "|---|---|---:|---|---|"]
for _, r in dq.iterrows():
    lines.append(f"| {r.problem} | {r.field} | {r['count']:,} | {r.treatment} | {r.justification} |")
open("../docs/data_quality_report.md", "w").write("\n".join(lines) + "\n")
print("Wrote docs/data_quality_report.md")

Saved 73,601 curated rows to data/curated/inspections_curated.csv
Wrote docs/data_quality_report.md
